Overview of dataset

In [20]:
import os
import sys

import numpy as np

sys.path.insert(0, os.path.abspath(".."))

In [21]:
import datasets.config
import pandas as pd
import PIL

from data.data_loader import get_data_loader_CIFAR10, get_data_loader_CIFAR10C
from datasets import ClassLabel, Value, load_dataset

datasets.config.PIL_AVAILABLE = True

In [22]:
dataloader = get_data_loader_CIFAR10C(32)
dataset = dataloader.dataset.ds

In [23]:
print(dataset.features)
print(dataset.features["image"])
print(dataset.features["label"].names)
print(dataset.features["corruption_name"])
print(dataset.features["corruption_level"])

{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']), 'corruption_name': Value('string'), 'corruption_level': Value('int32')}
Image(mode=None, decode=True)
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Value('string')
Value('int32')


In [24]:
def summarise_feature(dataset, col):
    feature = dataset.features[col]
    series = dataset.to_pandas()[col]
    if isinstance(feature, ClassLabel):
        names = feature.names
        counts = series.value_counts().sort_index()
        print(f"\n{col} ({len(names)} classes)")
        for i, name in enumerate(names):
            print(f"  {name}: {counts.get(i, 0)}")
    else:
        unique = sorted(series.unique())
        counts = series.value_counts().sort_index()
        print(f"\n{col} ({len(unique)} unique values)")
        for val in unique:
            print(f"  {val}: {counts.get(val, 0)}")


for col in ["label", "corruption_name", "corruption_level"]:
    summarise_feature(dataset, col)


label (10 classes)
  airplane: 95000
  automobile: 95000
  bird: 95000
  cat: 95000
  deer: 95000
  dog: 95000
  frog: 95000
  horse: 95000
  ship: 95000
  truck: 95000

corruption_name (19 unique values)
  brightness: 50000
  contrast: 50000
  defocus_blur: 50000
  elastic_transform: 50000
  fog: 50000
  frost: 50000
  gaussian_blur: 50000
  gaussian_noise: 50000
  glass_blur: 50000
  impulse_noise: 50000
  jpeg_compression: 50000
  motion_blur: 50000
  pixelate: 50000
  saturate: 50000
  shot_noise: 50000
  snow: 50000
  spatter: 50000
  speckle_noise: 50000
  zoom_blur: 50000

corruption_level (5 unique values)
  1: 190000
  2: 190000
  3: 190000
  4: 190000
  5: 190000


# Corruption

In [25]:
dataloader = get_data_loader_CIFAR10(32, notebook=True)
dataset = dataloader.dataset

In [26]:
print(dataset.classes)
# print(dataset.features["corruption_name"])
# print(dataset.features["corruption_level"])

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [27]:
def summarise_feature_pytorch(dataset):
    counts = pd.Series(dataset.targets).value_counts().sort_index()
    print(f"\nlabel ({len(dataset.classes)} classes)")
    for i, name in enumerate(dataset.classes):
        print(f"  {name}: {counts.get(i, 0)}")

In [28]:
summarise_feature_pytorch(dataset)


label (10 classes)
  airplane: 5000
  automobile: 5000
  bird: 5000
  cat: 5000
  deer: 5000
  dog: 5000
  frog: 5000
  horse: 5000
  ship: 5000
  truck: 5000


# Okay so do corruption pipeline

In [29]:
# Generate every single corruption -> Save as a paraquet file
# Then aggregate into a single file

# That's what the data_loader will handle
from imagecorruptions import corrupt, get_corruption_names

avaliable_corruptions = get_corruption_names()

print("All corruptions to be applied: ", avaliable_corruptions)
print("Dataset Length: ", len(dataset))
print(dataset[0])


def apply_corruption(image):
    corrupted_images = []
    for corruption in avaliable_corruptions:
        for severity in range(5):
            corrupted = corrupt(
                image, corruption_name=corruption, severity=severity - 1
            )
            corrupted_images.append(corrupted)
    return corrupted_images

All corruptions to be applied:  ['gaussian_noise', 'shot_noise', 'impulse_noise', 'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur', 'snow', 'frost', 'fog', 'brightness', 'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
Dataset Length:  50000
(<PIL.Image.Image image mode=RGB size=32x32 at 0x7FC8C6356DB0>, 6)


In [31]:
all_corrupted_images = []
for image in dataset:
    img_np = np.array(image[0])
    corrupted_images = apply_corruption(img_np)
    all_corrupted_images.append(corrupted_images)

AttributeError: Expecting type(image) to be numpy.ndarray

In [ ]:
len(all_corrupted_images)
assert all_corrupted_images == 50000